# MaxCut on C4

The **Maximum Cut** problem partitions graph vertices into two sets to
maximise the number of edges crossing the partition. We solve it on the
4-cycle $0{-}1{-}2{-}3{-}0$ with D-Wave Ocean's simulated annealer and
compare the result to what Qiskit QAOA gives.

The BQM encodes MaxCut as: $E = -\sum_{(i,j)\in E} s_i s_j$ where
aligned spins ($s_i = s_j$) cost $-1$ and anti-aligned spins gain $+1$.

This notebook is self-contained. It does not import `max_cut.py`.

In [ ]:
import dimod
import matplotlib.pyplot as plt
import neal
import networkx as nx
import numpy as np
import qiskit as qk
import scipy as scp

In [ ]:
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0)]
N = 4


def cut_size(bits):
    colors = [int(b) for b in bits[::-1]]
    return sum(colors[i] != colors[j] for i, j in EDGES)


bqm = dimod.BinaryQuadraticModel("SPIN")
for i, j in EDGES:
    bqm.add_variable(i, 0.0)
    bqm.add_variable(j, 0.0)
    bqm.add_interaction(i, j, -1.0)

print(f"BQM: {len(bqm.variables)} variables, {len(bqm.quadratic)} interactions")
print(f"Linear:     {dict(bqm.linear)}")
print(f"Quadratic:  {dict(bqm.quadratic)}")

## Solve with simulated annealing

In [ ]:
sampler = neal.SimulatedAnnealingSampler()
response = sampler.sample(bqm, num_reads=200, num_sweeps=500)
sample = response.first.sample
ocean_bits = "".join(str(sample.get(i, 0)) for i in range(N))
ocean_energy = response.first.energy
ocean_cut = cut_size(ocean_bits)

print(f"Best sample:  {ocean_bits}")
print(f"BQM energy:   {ocean_energy:+.2f}")
print(f"Cut value:    {ocean_cut}")

## Compare with Qiskit QAOA

In [ ]:
P = 2


def cost_layer(gamma):
    qc = qk.QuantumCircuit(N)
    for i, j in EDGES:
        qc.cx(i, j)
        qc.rz(2 * gamma, j)
        qc.cx(i, j)
    return qc


def mixer_layer(beta):
    qc = qk.QuantumCircuit(N)
    for q in range(N):
        qc.rx(2 * beta, q)
    return qc


def qaoa_circuit(params):
    qc = qk.QuantumCircuit(N)
    qc.h(range(N))
    for k in range(P):
        qc.compose(cost_layer(float(params[k])), inplace=True)
        qc.compose(mixer_layer(float(params[P + k])), inplace=True)
    return qc


def expected_cut(params):
    probs = qk.quantum_info.Statevector.from_instruction(
        qaoa_circuit(params)
    ).probabilities_dict()
    return sum(p * cut_size(b) for b, p in probs.items())


rng = np.random.default_rng(7)
guess = rng.uniform(0, np.pi, size=2 * P)
opt = scp.optimize.minimize(
    lambda p: -expected_cut(p),
    guess,
    method="COBYLA",
    options={"maxiter": 80, "rhobeg": 0.4},
)

probs = qk.quantum_info.Statevector.from_instruction(
    qaoa_circuit(opt.x)
).probabilities_dict()
qaoa_bits, qaoa_p = max(probs.items(), key=lambda kv: kv[1])
qaoa_cut = cut_size(qaoa_bits)

print(f"Most likely:  |{qaoa_bits}>")
print(f"Cut value:    {qaoa_cut}")
print(f"Probability:  {qaoa_p:.3f}")
print()
print(f"Ocean cut: {ocean_cut}   QAOA cut: {qaoa_cut}   Optimal: 4")

## Visualise the cut

Green edges cross the partition; dashed grey edges stay inside a partition.

In [ ]:
colors = [int(b) for b in ocean_bits[::-1]]
color_map = ["lightblue" if c == 0 else "salmon" for c in colors]
pos = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (0, 0)}
G = nx.cycle_graph(N)

fig, ax = plt.subplots(figsize=(5, 5))
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=color_map, node_size=800)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=14, font_weight="bold")
cut_edges = [(i, j) for i, j in EDGES if colors[i] != colors[j]]
uncut_edges = [(i, j) for i, j in EDGES if colors[i] == colors[j]]
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=cut_edges, width=3, edge_color="green")
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=uncut_edges, width=2, edge_color="gray", style="dashed")
ax.set_title(f"MaxCut on C4  cut={ocean_cut}")
ax.axis("off")
plt.tight_layout()
plt.show()